In [4]:
import pandas
import os 
import sys
from pathlib import Path
from dotenv import load_dotenv


In [5]:
from langchain_openai import OpenAI,ChatOpenAI
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent

C:\Users\tella\AppData\Local\Temp\ipykernel_46424\3691252223.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent


In [6]:
env_path = next(
    folder / ".env"
    for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / ".env").is_file()
)
load_dotenv(env_path, override=True)
OPENAI_APIKEY = os.getenv("OPENAI_APIKEY")

In [7]:
dataframes = [] 
loaded_names = []

In [8]:
os.listdir(".")

['csv_faq_agent.ipynb',
 'It_support_agent.ipynb',
 'llm_programatically_call.ipynb']

In [9]:
data_folder = Path("../Datasets")
csv_files = list(data_folder.glob("*.csv"))
print("Found CSV files:", csv_files)
print("Number of CSV files:", len(csv_files))

Found CSV files: [WindowsPath('../Datasets/credit_card_terms.csv'), WindowsPath('../Datasets/ecommerce_faqs.csv'), WindowsPath('../Datasets/hospital_policy.csv'), WindowsPath('../Datasets/saas_docs.csv')]
Number of CSV files: 4


In [10]:
try :
    for file_path in data_folder.glob("*.csv"):
        dataframe = pandas.read_csv(file_path)
        dataframes.append(dataframe)
        loaded_names.append(file_path.name)
        print(f"Loaded {file_path.name}: {len(dataframe)} rows")
except Exception as e:
    print(f"\nERROR loading files: {e}")
    sys.exit()


Loaded credit_card_terms.csv: 15 rows
Loaded ecommerce_faqs.csv: 15 rows
Loaded hospital_policy.csv: 15 rows
Loaded saas_docs.csv: 15 rows


In [11]:
print("Current folder:", os.getcwd())
print("CSV files:", loaded_names)
print("DataFrames:", len(dataframes))

Current folder: c:\Users\tella\OneDrive\Desktop\InterviewKickStart\py-frameworks\notebooks
CSV files: ['credit_card_terms.csv', 'ecommerce_faqs.csv', 'hospital_policy.csv', 'saas_docs.csv']
DataFrames: 4


In [12]:
system_prompt = """ You are helpful assistant capable of reading multiple CSV files and providing insights based on their content.
- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.
- When asked a question, determine which DataFrame is most relevant.
- Do NOT answer from general knowledge.
- Answer in plain English.   
"""



In [13]:
!pip install tabulate

In [14]:
import tabulate

In [15]:

try:
    
    # Initialize llm client use ChatOpenAI with model="gpt-4o-mini" and temperature=0.0
    llm_client = ChatOpenAI(api_key=OPENAI_APIKEY, model="gpt-4o-mini", temperature=0.0)
    agent = create_pandas_dataframe_agent(
        llm=llm_client,
        df=dataframes,
        system_prompt=system_prompt,
        loaded_names=loaded_names,
        verbose=True,
        allow_dangerous_code=True,
        agent_type="openai-functions"
    )
    
    print("\nAI Agent is ready! You can ask questions across ALL files.")
    print("Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'")

except Exception as e:
    print(f"Error initializing agent: {e}")
    


AI Agent is ready! You can ask questions across ALL files.
Example: 'What is the visiting hour in the hospital?' or 'What is the API limit?'


c:\Users\tella\OneDrive\Desktop\InterviewKickStart\py-frameworks\.venv\Lib\site-packages\langchain_experimental\agents\agent_toolkits\pandas\base.py:283: UserWarning: Received additional kwargs {'system_prompt': ' You are helpful assistant capable of reading multiple CSV files and providing insights based on their content.\n- You have access to 4 different datasets: SaaS Docs, Credit Card Terms, Hospital Policy, and Ecommerce FAQs.\n- When asked a question, determine which DataFrame is most relevant.\n- Do NOT answer from general knowledge.\n- Answer in plain English.   \n', 'loaded_names': ['credit_card_terms.csv', 'ecommerce_faqs.csv', 'hospital_policy.csv', 'saas_docs.csv']} which are no longer supported.
  warnings.warn(


In [ ]:
print("\nType 'exit' or 'quit' to stop conversation.\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    if not user_input.strip():
        continue
    
    final_query = system_prompt + "\n\nQuestion: " + user_input
    print(f"\n AI is Thinking ..")
    
    try :
        response = agent.run(final_query)
        print(f"AI: {response}\n" + "-"*30)
    except Exception as e:
        print(f"Error occurred while processing query: {e}\n")
   


Type 'exit' or 'quit' to stop conversation.


 AI is Thinking ..


> Entering new AgentExecutor chain...


C:\Users\tella\AppData\Local\Temp\ipykernel_46424\1391482647.py:14: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = agent.run(final_query)


The data present in the four datasets is as follows:

1. **Credit Card Terms (df1)**: This dataset contains information about various credit card terms, including details like the card type, category, specific terms (like APR and fees), and whether the rate is variable.

2. **Ecommerce FAQs (df2)**: This dataset includes frequently asked questions related to ecommerce, covering topics such as shipping, returns, warranties, and payment methods. Each entry has a question, an answer, related products, and the last updated date.

3. **Hospital Policy (df3)**: This dataset outlines various hospital policies, including topics like visiting hours, insurance verification, triage priority, medical records requests, and HIPAA compliance. Each policy has a unique ID, department, topic, policy text, effective date, and compliance level.

4. **SaaS Docs (df4)**: This dataset provides information about different features of a SaaS product, including API rate limits, user roles, data export capabilit